# Gold Fact Table Airport Performance 

In [8]:
from pyspark.sql.functions import col, sum, round, length

df_atfm    = spark.table("EAA_LakeHouse.silver.atfm_delays")
df_traffic = spark.table("EAA_LakeHouse.silver.airport_traffic")

print("=== ATFM schema ===")
df_atfm.printSchema()

print("=== Airport Traffic schema ===")
df_traffic.printSchema()

print(f"\nATFM rows    : {df_atfm.count()}")
print(f"Airport Traffic rows : {df_traffic.count()}")

StatementMeta(, c4e019eb-d4f7-4c45-95e7-5637d7f520ac, 10, Finished, Available, Finished, False)

=== ATFM schema ===
root
 |-- YEAR: integer (nullable = true)
 |-- MONTH_NUM: integer (nullable = true)
 |-- MONTH_MON: string (nullable = true)
 |-- FLT_DATE: date (nullable = true)
 |-- APT_ICAO: string (nullable = true)
 |-- APT_NAME: string (nullable = true)
 |-- STATE_NAME: string (nullable = true)
 |-- TOTAL_ARRIVALS: integer (nullable = true)
 |-- TOTAL_ATFM_DELAY_MINUTES: integer (nullable = true)
 |-- ACCIDENT_INCIDENT_DELAY_MINUTES: integer (nullable = true)
 |-- ATC_CAPACITY_DELAY_MINUTES: integer (nullable = true)
 |-- DEICING_DELAY_MINUTES: integer (nullable = true)
 |-- NON_ATC_EQUIPMENT_DELAY_MINUTES: integer (nullable = true)
 |-- AERODROME_CAPACITY_DELAY_MINUTES: integer (nullable = true)
 |-- ATC_INDUSTRIAL_ACTION_DELAY_MINUTES: integer (nullable = true)
 |-- AIRSPACE_MANAGEMENT_DELAY_MINUTES: integer (nullable = true)
 |-- NON_ATC_INDUSTRIAL_ACTION_DELAY_MINUTES: integer (nullable = true)
 |-- OTHER_DELAY_MINUTES: integer (nullable = true)
 |-- SPECIAL_EVENT_DELAY_MI

In [9]:
# Aggregate ATFM to airport × month
df_atfm_month = (
    df_atfm
    .groupBy("APT_ICAO", "year_month")
    .agg(
        sum("TOTAL_ATFM_DELAY_MINUTES").alias("total_atfm_delay_min"),
        sum("TOTAL_ARRIVALS").alias("atfm_monitored_arrivals"),
        sum("ATFM_DELAYED_ARRIVALS").alias("atfm_delayed_arrivals"),
        sum("ATFM_DELAYED_ARRIVALS_OVER_15_MINUTES").alias("atfm_delayed_over_15min"),
        sum("WEATHER_DELAY_MINUTES").alias("weather_delay_min"),
        sum("ATC_CAPACITY_DELAY_MINUTES").alias("atc_capacity_delay_min"),
        sum("ATC_STAFFING_DELAY_MINUTES").alias("atc_staffing_delay_min"),
        sum("AIRSPACE_MANAGEMENT_DELAY_MINUTES").alias("airspace_mgmt_delay_min"),
        sum("OTHER_DELAY_MINUTES").alias("other_delay_min")
    )
    .withColumn(
        "avg_atfm_delay_per_arrival",
        round(col("total_atfm_delay_min") / col("atfm_monitored_arrivals"), 2)
    )
)

# Aggregate Traffic to airport × month
df_traffic_month = (
    df_traffic
    .groupBy("airport_code", "year_month")
    .agg(
        sum("total_departures").alias("total_departures"),
        sum("total_arrivals").alias("total_arrivals"),
        sum("total_flight_movements").alias("total_flight_movements"),
        sum("ifr_departures").alias("ifr_departures"),
        sum("ifr_arrivals").alias("ifr_arrivals"),
        sum("ifr_flight_movements").alias("ifr_flight_movements")
    )
)

print(f"ATFM month rows    : {df_atfm_month.count()}")
print(f"Traffic month rows : {df_traffic_month.count()}")

StatementMeta(, c4e019eb-d4f7-4c45-95e7-5637d7f520ac, 11, Finished, Available, Finished, False)

ATFM month rows    : 20676
Traffic month rows : 23607


In [10]:
from pyspark.sql.functions import col, sum, round, greatest, when, coalesce, lit

# Traffic drives — left join ATFM
df_fact = (
    df_traffic_month.alias("trf")
    .join(
        df_atfm_month.alias("atfm"),
        on=(col("trf.airport_code") == col("atfm.APT_ICAO")) &
           (col("trf.year_month")   == col("atfm.year_month")),
        how="left"
    )
    .select(
        col("trf.airport_code").alias("apt_icao"),
        col("trf.year_month"),
        col("trf.total_departures"),
        col("trf.total_arrivals"),
        col("trf.total_flight_movements"),
        col("trf.ifr_departures"),
        col("trf.ifr_arrivals"),
        col("trf.ifr_flight_movements"),
        col("atfm.total_atfm_delay_min"),
        col("atfm.atfm_monitored_arrivals"),
        col("atfm.atfm_delayed_arrivals"),
        col("atfm.atfm_delayed_over_15min"),
        col("atfm.avg_atfm_delay_per_arrival"),
        col("atfm.weather_delay_min"),
        col("atfm.atc_capacity_delay_min"),
        col("atfm.atc_staffing_delay_min"),
        col("atfm.airspace_mgmt_delay_min"),
        col("atfm.other_delay_min")
    )
)

# Add dominant delay cause (only meaningful when ATFM data exists and total delay > 0)
cause_cols = [
    col("weather_delay_min"),
    col("atc_capacity_delay_min"),
    col("atc_staffing_delay_min"),
    col("airspace_mgmt_delay_min"),
    col("other_delay_min")
]

df_fact = df_fact.withColumn(
    "dominant_delay_cause",
    when(col("total_atfm_delay_min").isNull() | (col("total_atfm_delay_min") == 0), None)
    .when(col("weather_delay_min")       == greatest(*cause_cols), lit("WEATHER"))
    .when(col("atc_capacity_delay_min")  == greatest(*cause_cols), lit("ATC_CAPACITY"))
    .when(col("atc_staffing_delay_min")  == greatest(*cause_cols), lit("ATC_STAFFING"))
    .when(col("airspace_mgmt_delay_min") == greatest(*cause_cols), lit("AIRSPACE_MGMT"))
    .otherwise(lit("OTHER"))
)

matched   = df_fact.filter(col("total_atfm_delay_min").isNotNull()).count()
unmatched = df_fact.filter(col("total_atfm_delay_min").isNull()).count()

print(f"Total rows         : {df_fact.count()}")
print(f"Matched (ATFM)     : {matched}")
print(f"Unmatched (no ATFM): {unmatched}")
display(df_fact.orderBy(col("total_atfm_delay_min").desc_nulls_last()).limit(10))

StatementMeta(, c4e019eb-d4f7-4c45-95e7-5637d7f520ac, 12, Finished, Available, Finished, False)

Total rows         : 23607
Matched (ATFM)     : 20676
Unmatched (no ATFM): 2931


SynapseWidget(Synapse.DataFrame, 06f9ddf0-3b0a-4ce4-a3b5-d830cdaad529)

In [11]:
# Write to Lakehouse gold schema
df_fact.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.fact_airport_performance_month")

print(f"fact_airport_performance_month written : {df_fact.count()} rows")

StatementMeta(, c4e019eb-d4f7-4c45-95e7-5637d7f520ac, 13, Finished, Available, Finished, False)

fact_airport_performance_month written : 23607 rows


In [14]:
from pyspark.sql.functions import min as spark_min, max as spark_max



df = spark.table("gold.fact_airport_performance_month")
errors = []

# CRITICAL — no null grain keys
for c in ["apt_icao", "year_month"]:
    n = df.filter(col(c).isNull()).count()
    if n > 0:
        errors.append(f"FAIL: {n} null {c}")

# CRITICAL — no duplicate grain
dupes = df.groupBy("apt_icao", "year_month").count().filter(col("count") > 1).count()
if dupes > 0:
    errors.append(f"FAIL: {dupes} duplicate grain rows")

# CRITICAL — avg delay must be non-negative when present
neg_delay = df.filter(col("avg_atfm_delay_per_arrival").isNotNull() & (col("avg_atfm_delay_per_arrival") < 0)).count()
if neg_delay > 0:
    errors.append(f"FAIL: {neg_delay} rows with negative avg_atfm_delay_per_arrival")

# CRITICAL — dominant_delay_cause valid values
valid_causes = {"WEATHER", "ATC_CAPACITY", "ATC_STAFFING", "AIRSPACE_MGMT", "OTHER"}
invalid_cause = df.filter(col("dominant_delay_cause").isNotNull() & ~col("dominant_delay_cause").isin(valid_causes)).count()
if invalid_cause > 0:
    errors.append(f"FAIL: {invalid_cause} invalid dominant_delay_cause values")

# INFO — ATFM coverage rate
matched = df.filter(col("total_atfm_delay_min").isNotNull()).count()
total   = df.count()
print(f"ATFM coverage : {matched}/{total} ({matched/total*100:.1f}%)")

# INFO — year_month range
display(df.agg(spark_min("year_month").alias("min_ym"), spark_max("year_month").alias("max_ym")))

if errors:
    raise ValueError("\n".join(errors))
else:
    print(f"All checks passed — {total} rows")

StatementMeta(, c4e019eb-d4f7-4c45-95e7-5637d7f520ac, 16, Finished, Available, Finished, False)

ATFM coverage : 20676/23607 (87.6%)


SynapseWidget(Synapse.DataFrame, 15ef4c2b-5e64-4d04-ac74-4f0253ae4e7b)

All checks passed — 23607 rows
